# CHD-CXR VLM Evaluation Pipeline

Benchmarks HuggingFace vision-language models on pediatric congenital heart disease
classification from chest X-rays (CHD-CXR dataset).

**Before running:**
1. Set runtime to GPU: *Runtime → Change runtime type → A100 (or T4 for smaller models)*
2. Add your HF token as a Colab Secret: *left sidebar → 🔑 Secrets → Add `HF_TOKEN`*
3. Make sure you have accepted the licence for any gated models on huggingface.co

**Models referenced in the research plan:**
| Model | HF identifier | Gated? |
|---|---|---|
| MedGemma 4B | `google/medgemma-4b-it` | Yes |
| LLaVA-Med | `microsoft/llava-med-v1.5-mistral-7b` | No |
| BiomedCLIP (feature ref) | `microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224` | No |


In [10]:
!pip install -q --upgrade "transformers>=4.40.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 143.6 MB/s eta 0:00:00


## 0 · Environment setup

In [1]:
import subprocess, sys

# Check GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT FOUND — switch to a GPU runtime')

GPU: NVIDIA A100-SXM4-40GB, 40960 MiB


In [12]:
# Clone repo and install dependencies
!git clone https://github.com/rushankgoyal/chd-eval.git
%cd chd-eval
!pip install -q -r requirements.txt

Cloning into 'chd-eval'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 21 (delta 7), reused 13 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 61.05 KiB | 20.35 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/chd-eval/chd-eval


## 1 · HuggingFace authentication

Token is read from Colab Secrets — never hardcoded.

In [3]:
from google.colab import userdata
from huggingface_hub import login, whoami

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print('Authenticated as:', whoami()['name'])

Authenticated as: rushankg


## 2 · Dataset

The CHD-CXR dataset (828 pediatric chest X-rays, 4 classes) is available from the
original paper. Upload it to this session or mount Google Drive.

**Expected directory structure:**
```
data/
  ASD/   image001.png  ...
  VSD/   image002.png  ...
  PDA/   image003.png  ...
  Normal/ image004.png ...
```

In [7]:
# Option A: mount Google Drive (recommended for large datasets)
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_ROOT = '/content/drive/MyDrive/chd-cxr'  # adjust path

# Option B: upload a zip directly to Colab (comment out Option A first)
# from google.colab import files
# uploaded = files.upload()          # select your chd-cxr.zip
# !unzip -q chd.zip -d data/
DATASET_ROOT = 'data'

In [8]:
from evaluate import load_samples_from_directory, LABELS

samples = load_samples_from_directory(DATASET_ROOT)
print(f'Loaded {len(samples)} samples')
for label in LABELS:
    n = sum(s.label == label for s in samples)
    print(f'  {label}: {n}')

Loaded 828 samples
  ASD: 194
  VSD: 210
  PDA: 216
  Normal: 208


## 3 · Configuration

Edit this cell to choose which models and prompts to evaluate.
For an ESR abstract, two models × two prompts is a practical starting point.

In [13]:
from evaluate import PROMPTS

# ── Models to evaluate ────────────────────────────────────────────────────
# Comment/uncomment as needed. Each model is loaded, evaluated, then unloaded.
MODELS = [
    'google/medgemma-4b-it',          # requires licence acceptance
    'microsoft/llava-med-v1.5-mistral-7b',
]

# ── Prompting strategies ──────────────────────────────────────────────────
# Keys must match entries in evaluate.PROMPTS
PROMPT_IDS = ['ZSD', 'CoT']

# ── Inference settings ────────────────────────────────────────────────────
LOAD_IN_4BIT  = False   # set True to halve VRAM usage (slight accuracy cost)
MAX_NEW_TOKENS = 512    # 256 is enough for ZSD/RCE; 512 recommended for CoT
RESULTS_DIR    = 'results'

import os; os.makedirs(RESULTS_DIR, exist_ok=True)
print('Models:', MODELS)
print('Prompts:', PROMPT_IDS)

Models: ['google/medgemma-4b-it', 'microsoft/llava-med-v1.5-mistral-7b']
Prompts: ['ZSD', 'CoT']


## 4 · Run evaluation

Iterates over every (model, prompt) pair. Each model is fully unloaded from GPU
memory before the next one is loaded, keeping peak VRAM manageable.
Partial results are saved after each (model, prompt) run so progress is never lost.

In [21]:
from content/chd-eval/evaluate.py import CHDEvaluator

SyntaxError: invalid syntax (3140248352.py, line 1)

In [22]:
import gc, torch, pandas as pd
from pathlib import Path
import sys
sys.path.append("/content/chd-eval")

from evaluate import CHDEvaluator, PROMPTS

all_results = []

for model_name in MODELS:
    print(f'\n{"="*60}')
    print(f'Model: {model_name}')
    print(f'{"="*60}')

    evaluator = CHDEvaluator(
        model_name,
        hf_token=HF_TOKEN,
        load_in_4bit=LOAD_IN_4BIT,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    for prompt_id in PROMPT_IDS:
        print(f'\n  Prompt: {prompt_id}')
        df = evaluator.evaluate(samples, prompt=PROMPTS[prompt_id], prompt_id=prompt_id)

        # Save partial results immediately
        safe_model = model_name.replace('/', '__')
        out_path = Path(RESULTS_DIR) / f'{safe_model}__{prompt_id}.csv'
        df.to_csv(out_path, index=False)
        print(f'  Saved → {out_path}')

        parse_rate = df['parse_success'].mean()
        print(f'  Parse rate: {parse_rate:.1%}')
        all_results.append(df)

    # Unload model to free VRAM before loading next one
    del evaluator
    gc.collect()
    torch.cuda.empty_cache()
    print(f'  VRAM freed.')

# Combine all runs into a single DataFrame
results_df = pd.concat(all_results, ignore_index=True)
results_df.to_csv(f'{RESULTS_DIR}/all_results.csv', index=False)
print(f'\nAll results saved to {RESULTS_DIR}/all_results.csv')
print(f'Total rows: {len(results_df)}')


Model: google/medgemma-4b-it


AttributeError: module transformers has no attribute AutoModelForVision2Seq

## 5 · Analysis

In [ ]:
from analyze import run_full_analysis, print_summary

# Load from saved CSV if restarting from a saved run:
# results_df = pd.read_csv(f'{RESULTS_DIR}/all_results.csv')

analysis = run_full_analysis(results_df, n_bootstrap=2000)
print_summary(analysis)

In [ ]:
# Inspect the full metrics table
analysis['metrics']

In [ ]:
# Per-class breakdown
analysis['per_class_metrics'].pivot_table(
    index=['model_name', 'prompt_id'], columns='class', values='f1'
).round(3)

In [ ]:
# Pairwise significance tests (BH-corrected McNemar)
analysis['mcnemar_results']

## 6 · Visualisation

In [ ]:
import matplotlib.pyplot as plt
from visualize import (
    plot_holistic_dashboard,
    plot_macro_f1_bar,
    plot_per_class_heatmap,
    plot_bootstrap_ci,
    plot_all_confusion_matrices,
    save_all_figures,
)

# Holistic dashboard — the single figure for an abstract/poster
fig = plot_holistic_dashboard(analysis, results_df=results_df)
plt.show()

In [ ]:
fig = plot_macro_f1_bar(analysis)
plt.show()

In [ ]:
fig = plot_per_class_heatmap(analysis)
plt.show()

In [ ]:
fig = plot_bootstrap_ci(analysis)
plt.show()

In [ ]:
fig = plot_all_confusion_matrices(analysis)
plt.show()

In [ ]:
# Save all figures as PDFs to the figures/ directory, then download
save_all_figures(analysis, results_df=results_df, output_dir='figures', fmt='pdf')

# Zip and download everything
!zip -r results_and_figures.zip results/ figures/
from google.colab import files
files.download('results_and_figures.zip')